In [3]:
import pandas as pd
# Load
apphistory = pd.read_csv('../data/Opportunityapplicationhistorytest.csv', index_col=False)
apphistory.head()

C:\Users\juanb\AppData\Local\Temp\ipykernel_24304\296490263.py:3: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  apphistory = pd.read_csv('../data/Opportunityapplicationhistorytest.csv', index_col=False)


,Opportunity Application Id,Opportunity Id,GeneratedPlacementId,Opportunity Name,Opportunity Status,Eligible Student Requirement Groups,Min Places,Max Places,Agency Id,Agency Name,Latitude,Longitude,Student Code,Opportunity Application Status,Opportunity Application Status _update Date,Opportunity Application Status _update User
0,7e808278-2d06-4067-dcab-08dea10975a7,aa60714d-5a92-422a-e09b-08de90cb26f9,16184,Marketing & Communications Intern - CarbonCLAIR,Published,2026 Career Launch - Marketing and Communications,1,3,d7a3b918-5017-f011-81a3-90cb95efafa1,CarbonCLAIR,40.821691,-73.947945,24575565,Placed,06/09/2026 11:19 AM,elijahnunez@gmail.com
1,7e808278-2d06-4067-dcab-08dea10975a7,aa60714d-5a92-422a-e09b-08de90cb26f9,16184,Marketing & Communications Intern - CarbonCLAIR,Published,2026 Career Launch - Marketing and Communications,1,3,d7a3b918-5017-f011-81a3-90cb95efafa1,CarbonCLAIR,40.885286,-73.906933,24575565,Placed,06/09/2026 11:19 AM,elijahnunez@gmail.com
2,07968675-3e0f-4abb-dcbc-08dea10975a7,29951042-89d1-4708-e01f-08de90cb26f9,15886,Python/Django Coding Internship,Published,2026 Career Launch - STEM and Green,7,10,fc9db918-5017-f011-81a3-90cb95efafa1,PYE Education Center,40.744350,-73.890260,24603555,Unsuccessful,06/09/2026 10:32 AM,elijahnunez@gmail.com
3,68d5af6c-101d-44d3-69d5-08dea5adcd1a,dc47a011-5a0c-4ce3-a6db-08de6fd96e5d,16185,Social Media Intern – The Difference App,Published,2026 Career Launch - Marketing and Communications,8,10,439eb918-5017-f011-81a3-90cb95efafa1,The Difference App LLC,40.900848,-73.785407,24283815,Placed,06/09/2026 11:19 AM,elijahnunez@gmail.com
4,fc8d8cdd-fbf5-4ca0-69db-08dea5adcd1a,8b34fb91-8219-4524-5638-08de6801b4ac,16186,"Research Intern - SAL Consulting, LLC",Published,2026 Career Launch - Marketing and Communications,1,1,b66788dc-9fec-45ec-04ad-08de64be232d,"SAL Consulting, LLC",NaN,NaN,24454578,Placed,06/09/2026 11:19 AM,elijahnunez@gmail.com


In [2]:
"""
Career Launch 2026 - Program at a Glance
-----------------------------------------
Rebuilds the "Program at a Glance" numbers from the Opportunity Application
History export, anchored on the corrected unique-student count (~1907),
instead of the old stale 1,816 figure.

Data quality note baked into this script:
The raw export has a lat/long join fan-out - the same
Opportunity Application Id can appear multiple times with different
Latitude/Longitude values but identical everything else. We drop that
duplication first so all downstream counts are per-application, not
per-(application x location).
"""

import pandas as pd

INPUT_PATH = "../data/Opportunityapplicationhistorytest.csv"

# Bulk window narrowed per stakeholder revision: only June 9-11 counts as
# the bulk window now. Everything before it is Pre-Match, even if made by
# Elijah. Everything after it is Rematch (unchanged).
BULK_WINDOW_START = pd.Timestamp("2026-06-09")
BULK_WINDOW_END = pd.Timestamp("2026-06-11")


def load_and_dedupe(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, index_col=False)
    print(f"raw rows: {len(df)}")

    # Drop the lat/long fan-out: everything except Latitude/Longitude
    # defines a unique application record.
    key_cols = [c for c in df.columns if c not in ("Latitude", "Longitude")]
    df = df.drop_duplicates(subset=key_cols).copy()
    print(f"rows after dropping lat/long fan-out duplicates: {len(df)}")

    df["upd_date"] = pd.to_datetime(
        df["Opportunity Application Status _update Date"], errors="coerce"
    )
    return df


def classify_match_type(row: pd.Series) -> str:
    """
    Final classification (v3, per stakeholder revision):

    Bulk Match:   updated June 9-11 (inclusive) AND by Elijah
                  (elijahnunez@gmail.com).
    Manual Match: updated June 9-11 but NOT by Elijah.
    Pre-Match:    updated before June 9 - regardless of creator (this is the
                  change: an Elijah-updated record before June 9 is now
                  Pre-Match, not Bulk).
    Rematch:      updated after June 11. Unchanged from before. Cross-
                  reference only - a separate CSV is the source of truth
                  for rematches, this is NOT the official rematch count.
    """
    date = row["upd_date"]
    if pd.isna(date):
        return "Unknown"
    d = date.normalize()

    if d > BULK_WINDOW_END:
        return "Rematch"
    elif d < BULK_WINDOW_START:
        return "Pre-Match"

    # d is within June 9-11 inclusive
    creator = str(row["Opportunity Application Status _update User"]).strip().lower()
    return "Bulk Match" if creator == "elijahnunez@gmail.com" else "Manual Match"


def program_at_a_glance(df: pd.DataFrame) -> dict:
    total_unique_students = df["Student Code"].nunique()

    placed = df[df["Opportunity Application Status"] == "Placed"].copy()
    # Keep each student's most recent Placed record only, so students with
    # a genuine rematch (2nd distinct placement) aren't double-counted
    # across match-type buckets.
    placed_latest = (
        placed.sort_values("upd_date")
        .drop_duplicates(subset="Student Code", keep="last")
    )
    placed_latest["match_type"] = placed_latest.apply(classify_match_type, axis=1)

    unsuccessful = df[df["Opportunity Application Status"] == "Unsuccessful"]

    creator = df["Opportunity Application Status _update User"].apply(
        lambda x: "Elijah Nunez"
        if str(x).strip().lower() == "elijahnunez@gmail.com"
        else "Other/Student-attributed"
    )

    stats = {
        "unique_students_sent_confirmation": total_unique_students,
        "unique_students_placed": placed_latest["Student Code"].nunique(),
        "unique_agencies_matched": placed_latest["Agency Name"].nunique(),
        "match_type_breakdown": placed_latest["match_type"].value_counts().to_dict(),
        "unsuccessful_total_rows": len(unsuccessful),
        "unsuccessful_unique_students": unsuccessful["Student Code"].nunique(),
        "unsuccessful_by_opportunity_status": unsuccessful["Opportunity Status"]
        .value_counts()
        .to_dict(),
        "record_creator_breakdown": creator.value_counts().to_dict(),
    }
    return stats, placed_latest


def hub_breakdown(df: pd.DataFrame, placed_latest: pd.DataFrame) -> pd.DataFrame:
    hub_col = "Eligible Student Requirement Groups"
    rows = []
    for hub, hub_df in df.groupby(hub_col):
        hub_placed = placed_latest[placed_latest[hub_col] == hub]
        rows.append(
            {
                "Hub": hub,
                "Unique Students": hub_df["Student Code"].nunique(),
                "Unique Agencies": hub_placed["Agency Name"].nunique(),
                "Pre-Match": (hub_placed["match_type"] == "Pre-Match").sum(),
                "Bulk Match": (hub_placed["match_type"] == "Bulk Match").sum(),
                "Manual Match": (hub_placed["match_type"] == "Manual Match").sum(),
                "Rematch (x-ref only)": (hub_placed["match_type"] == "Rematch").sum(),
            }
        )
    out = pd.DataFrame(rows)

    # Don't just sum the per-hub "Unique Agencies" column for the Total row -
    # an agency posting opportunities across more than one hub would get
    # counted once per hub, inflating the total. Recompute it directly across
    # the whole placed population instead. Unique Students/Pre/Bulk/Manual are
    # safe to sum because each student only belongs to one hub.
    total = out.drop(columns="Hub").sum(numeric_only=True)
    total["Unique Agencies"] = placed_latest["Agency Name"].nunique()
    total["Hub"] = "Total"
    out = pd.concat([out, pd.DataFrame([total])], ignore_index=True)
    return out


if __name__ == "__main__":
    df = load_and_dedupe(INPUT_PATH)
    stats, placed_latest = program_at_a_glance(df)

    print("\n=== Program at a Glance ===")
    for k, v in stats.items():
        print(f"{k}: {v}")

    print("\n=== Hub Breakdown ===")
    print(hub_breakdown(df, placed_latest).to_string(index=False))

raw rows: 2969
rows after dropping lat/long fan-out duplicates: 2075

=== Program at a Glance ===
unique_students_sent_confirmation: 1907
unique_students_placed: 1770
unique_agencies_matched: 325
match_type_breakdown: {'Bulk Match': 1114, 'Pre-Match': 389, 'Rematch': 140, 'Manual Match': 127}
unsuccessful_total_rows: 297
unsuccessful_unique_students: 273
unsuccessful_by_opportunity_status: {'Published': 279, 'Archived': 10, 'Draft': 8}
record_creator_breakdown: {'Elijah Nunez': 1315, 'Other/Student-attributed': 760}

=== Hub Breakdown ===
                                               Hub  Unique Students  Unique Agencies  Pre-Match  Bulk Match  Manual Match  Rematch (x-ref only)
2026 Career Launch - Community and Social Services              469              113        138         242            34                    28
                   2026 Career Launch - Healthcare              458               94        101         264            25                    55
 2026 Career Launch - M

C:\Users\juanb\AppData\Local\Temp\ipykernel_19188\4001775489.py:28: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(path, index_col=False)
